Code to plot the data and best fit model using Mulens Model

You may have to install matplotlib if you don't have it already!

In [ ]:
import MulensModel as mm
import matplotlib.pyplot as plt

In [ ]:
#Start with the data frame lc_data that has the light curve data for the event
#first extract all fit parameters from your data file and store them in a dictionary called fit_params
fit_params = {}

#then crop lc_data to only include data between t0 - 3*tE and t0 + 3*tE
mask = (lc_data['bjd'] > fit_params['t_0'] - 3*fit_params['t_E']) & (lc_data['bjd'] < fit_params['t_0'] + 3*fit_params['t_E'])

#Let us create a MulensData object for the cropped data
data_event_1 = mm.MulensData(data_list= [lc_data['bjd'][mask], lc_data['mag'][mask], lc_data['mag_err'][mask]], #a list with 3 lists/arrays for time, mag, err in that order
                             phot_fmt='mag',
                             chi2_fmt = 'flux',
                             plot_properties={'color': '#a859e4',
                                                 'fmt': 'o',
                                                 'markersize': 1,
                                                 'show_errorbars': True,
                                                 'label': 'Roman W149'
                                                 }
                               )

#Then we create a model object
#Create a PSPL model
params_pspl = {'t_0': fit_params['t_0'], 'u_0': fit_params['u_0'], 't_E': fit_params['t_E']}
model_pspl = mm.Model(params_pspl) 
model_pspl.set_magnification_methods([data_event_1.time.min(), 'point_source', data_event_1.time.max()])

#Create an event object
event_roman = mm.Event(datasets=[data_event_1], model=model_pspl)

chi2 = event_roman.get_chi2()
FS, FB = event_roman.get_ref_fluxes()

#Plot the data and initial model
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={'height_ratios': [3, 1]},  sharex=True)
plt.sca(ax1)
data_event_1.plot(phot_fmt='flux', subtract_2450000=True, label=fr'$\chi^2 = {chi2:.2f}$, Reduced $\chi^2 = {chi2/(len(data_event_1.flux) - 5):.2f}$')
model_mag = model_pspl.get_magnification(data_event_1.time)
model_flux = FS * model_mag + FB
plt.plot(data_event_1.time - 2450000, model_flux, color='black', label='Best fit PSPL model')
#Plot the residuals
plt.sca(ax2)
plt.errorbar(data_event_1.time - 2450000, data_event_1.flux - model_flux, yerr=data_event_1.err_flux, color='#a859e4', label='Residuals')
plt.axhline(0, color='black', linestyle='-')
ax1.set_ylabel('Flux')
ax2.set_xlabel('HJD - 2450000')
ax2.set_ylabel('Residuals')
ax1.legend()
fig.tight_layout()
plt.show()